# Yahoo Fantasy Baseball 2

Thin notebook — all logic lives in `yahoo.py`.
Uses the `Yahoo` class and helper functions from `yahoo.py` to manage your fantasy team via the Yahoo Fantasy Sports API.

**Credentials required in `.env`:**
```
Client_ID:      "your_yahoo_app_client_id"
Client_Secret:  "your_yahoo_app_client_secret"
Yahoo_Email:    "your_yahoo_account_email"
Yahoo_Password: "your_yahoo_account_password"
```

Section 1 handles OAuth automatically — no manual steps needed.  
On subsequent runs, saved tokens are reused automatically.

## Section 1: Setup & Auth

In [ ]:
import importlib
import logging

import pandas as pd
pd.set_option('display.max_columns', 15)
pd.set_option('display.width', 120)

logging.getLogger('yahoo_oauth').setLevel(logging.WARNING)

import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('__file__')), '..'))

import yahoo as _yahoo_mod
importlib.reload(_yahoo_mod)
from yahoo import Yahoo, init_auth, top_available_all_leagues

SEASON     = 2026
CREDS_FILE = init_auth()  # reads .env, handles OAuth, returns creds_file path

Credentials written to browser/yahoo_oauth.json — tokens present
Refresh token found — no browser needed.
OAuth OK — session ready


## Section 2: Find Your League ID

Lists all your active MLB fantasy leagues. Copy the `league_id` for your team into the config cell below.

In [2]:
leagues_df = Yahoo.list_leagues(CREDS_FILE, SEASON)
print(f'Found {len(leagues_df)} MLB league(s) for {SEASON}:')
leagues_df

Found 8 MLB league(s) for 2026:


,league_id,team_name,name,scoring_method,num_teams,draft_status,league_key
0,18397,Mother Tuckers 100,Yahoo Prize H2H-Cat 18397,categories,12,postdraft,469.l.18397
1,41036,Dr. StrangeGlove 100,Yahoo Prize H2H-Cat 41036,categories,12,postdraft,469.l.41036
2,15981,Suave 100,Yahoo Prize Roto 15981,roto,12,postdraft,469.l.15981
3,68331,Japanese Kokamo Okamo 100,Yahoo Prize H2H-Cat 68331,categories,12,postdraft,469.l.68331
4,148341,Schlittler 100,Yahoo Prize H2H-Cat 148341,categories,12,postdraft,469.l.148341
5,3924,My Bradish Burns 100,Yahoo Prize H2H-Cat 3924,categories,12,postdraft,469.l.3924
6,690,Big Dick 000,Yahoo H2H-Cat 690,categories,10,postdraft,469.l.690
7,156186,Witty Gunnars 100,Yahoo Prize Roto 156186,roto,12,postdraft,469.l.156186


## Section 3: Config — set your league ID

In [3]:
# ── Set this from the output above ───────────────────────────────────────────
LEAGUE_ID = leagues_df['league_id'].iloc[0]   # auto-picks first league; change if you have multiple
# LEAGUE_ID = '12345'                          # or hardcode here
# ─────────────────────────────────────────────────────────────────────────────

print(f'League: {leagues_df.loc[leagues_df.league_id == LEAGUE_ID, "name"].iloc[0]}')
print(f'ID:     {LEAGUE_ID}')

League: Yahoo Prize H2H-Cat 18397
ID:     18397


## Section 4: Initialize Yahoo Client

In [4]:
yf = Yahoo(league_id=LEAGUE_ID, season=SEASON, creds_file=CREDS_FILE).fetch()
print(yf)

Yahoo(league=469.l.18397, team=469.l.18397.t.9)


## Section 5: Your Team

In [6]:
# Current roster
roster = yf.roster
print(f'Roster ({len(roster)} players):')
roster

Roster (23 players):


,slot,name,team,positions,status,player_key
0,1B,Rafael Devers,SF,1B,,469.p.10235
1,2B,Xavier Edwards,MIA,"2B,SS",,469.p.11375
2,3B,Matt Chapman,SF,3B,,469.p.10205
3,BN,Victor Scott II,STL,OF,,469.p.61888
4,BN,Josh Hader,HOU,RP,DTD,469.p.10131
5,BN,Michael Wacha,KC,SP,,469.p.9329
6,BN,Andrew Painter,PHI,SP,,469.p.60592
7,BN,Sean Manaea,NYM,"SP,RP",,469.p.9582
8,BN,Will Warren,NYY,SP,,469.p.60295
9,C,Yainer Diaz,HOU,C,,469.p.12435


In [7]:
# Current matchup
m = yf.matchup
if m:
    print(f"Week {m['week']}  ({m['start']} \u2192 {m['end']})")
    print(f"  You:      {m.get('you', '\u2014')}")
    print(f"  Opponent: {m.get('opponent', '\u2014')}")
else:
    print('No active matchup found (offseason or between weeks)')

Week 1  (2026-03-25 → 2026-03-29)
  You:      Mother Tuckers 100
  Opponent: $100 Standard


In [8]:
# Opponent's roster
if m:
    opp_roster = yf.opponent_roster
    print(f"Opponent roster \u2014 {m.get('opponent', '\u2014')} ({len(opp_roster)} players):")
    display(opp_roster)
else:
    print('No active matchup.')

Opponent roster — $100 Standard (23 players):


,slot,name,team,positions,status,player_key
0,1B,Vinnie Pasquantino,KC,1B,,469.p.12449
1,2B,Jose Altuve,HOU,"2B,OF",,469.p.8996
2,3B,Manny Machado,SD,3B,,469.p.9111
3,BN,Noelvi Marte,CIN,"3B,OF",,469.p.11529
4,BN,Ezequiel Tovar,COL,SS,,469.p.12322
5,BN,Brenton Doyle,COL,OF,,469.p.12136
6,BN,Jonathan Aranda,TB,1B,,469.p.12309
7,BN,Sandy Alcantara,MIA,SP,,469.p.10597
8,BN,Joe Musgrove,SD,SP,,469.p.10185
9,OF,Fernando Tatis Jr.,SD,OF,,469.p.10639


## Section 6: League Standings

In [9]:
yf.standings

,rank,team,wins,losses,ties,pct,gb
0,1,Cruz’s Cool Crew,0,0,0,0.0,-
1,2,Underdog,0,0,0,0.0,-
2,3,Blizzard 💯 ⚾️,0,0,0,0.0,-
3,4,New York Yankees,0,0,0,0.0,-
4,5,$100 Standard,0,0,0,0.0,-
5,6,Benjamin Bombers,0,0,0,0.0,-
6,7,Couch Hundy,0,0,0,0.0,-
7,8,Henchmen,0,0,0,0.0,-
8,9,Mother Tuckers 100,0,0,0,0.0,-
9,10,Bronx Bombers,0,0,0,0.0,-


## Section 7: Waiver Wire & Free Agents

In [10]:
# All players on waivers (paginated — fetches every player)
waivers = yf.waivers()
print(f'Waiver wire ({len(waivers)} players):')
waivers

Waiver wire (2070 players):


,name,team,positions,status,player_key
0,Masyn Winn,STL,SS,,469.p.12025
1,Kerry Carpenter,DET,OF,,469.p.12667
2,Ramón Laureano,SD,OF,DTD,469.p.11124
3,Andrew Vaughn,MIL,1B,,469.p.11866
4,Daulton Varsho,TOR,OF,,469.p.10843
...,...,...,...,...,...
2065,Maverick Handley,BAL,C,NA,469.p.60901
2066,Bradley Blalock,MIA,SP,,469.p.63311
2067,Jack Kochanowicz,LAA,SP,,469.p.11774
2068,Germán Márquez,SD,SP,,469.p.10402


In [11]:
# All free agents (paginated — fetches every player)
# Note: many waiver-only leagues return 0 free agents; all players are on waivers instead
fa = yf.free_agents()
print(f'Free agents: {len(fa)} player(s)')
fa

Free agents: 0 player(s)


,name,team,positions,status,player_key


In [12]:
# Filter by position — change 'SP' to any Yahoo position code
# Positions: C  1B  2B  3B  SS  OF  Util  SP  RP  P
waivers_sp = yf.waivers(position='SP')
print(f'SP on waivers ({len(waivers_sp)}):')
waivers_sp

SP on waivers (635):


,name,team,positions,status,player_key
0,Roki Sasaki,LAD,SP,,469.p.64803
1,José Soriano,LAA,SP,,469.p.11327
2,Reynaldo López,ATL,SP,,469.p.10152
3,Bailey Ober,MIN,SP,,469.p.12096
4,Mitch Keller,PIT,SP,,469.p.10565
...,...,...,...,...,...
630,Cal Quantrill,TEX,SP,NA,469.p.10573
631,Bradley Blalock,MIA,SP,,469.p.63311
632,Jack Kochanowicz,LAA,SP,,469.p.11774
633,Germán Márquez,SD,SP,,469.p.10402


In [13]:
# Search for a specific player (any ownership status)
yf.search('Pete Alonso')

,name,team,positions,status,player_key
0,Pete Alonso,BAL,1B,,469.p.10918


## Section 8: Add / Drop Players

Uncomment and edit the player names to execute transactions.

In [16]:
# Add a free agent
# yf.add('Pete Alonso')

# Add and simultaneously drop another player
# yf.add('Roki Sasaki', drop='Will Warren')

# FAAB leagues: include a bid amount
# yf.add('Roki Sasaki', drop='Joey Votto', faab=15)

# Drop only
# yf.drop('Joey Votto')

print('Uncomment a line above to execute a transaction.')

Uncomment a line above to execute a transaction.


## Section 9: Lineup Management

In [14]:
# Quick-reference: your bench and IL slots right now
bench = roster[roster['slot'].isin(['BN', 'IL', 'IL+', 'NA'])]
print('Bench / IL:')
print(bench[['slot', 'name', 'team', 'positions', 'status']].to_string(index=False))

Bench / IL:
slot            name team positions status
  BN Victor Scott II  STL        OF       
  BN   Michael Wacha   KC        SP       
  BN  Andrew Painter  PHI        SP       
  BN     Sean Manaea  NYM     SP,RP       
  BN     Will Warren  NYY        SP       


In [14]:
# ── Live example: bench Josh Hader ───────────────────────────────────────────
# Shows current slot, moves to BN, then confirms the change.

hader = roster[roster['name'].str.contains('Hader', case=False, na=False)]
print("Before:", hader[['slot', 'name', 'positions', 'status']].to_string(index=False))

yf.move('Josh Hader', 'P') 
# yf.bench('Josh Hader')

# ── Other lineup operations (uncomment to use) ────────────────────────────────
# yf.move('Josh Hader', 'P')          # move back to active P slot
# yf.move('Yainer Diaz', 'BN')        # move to a specific slot
# yf.il('Spencer Strider')            # move to IL
# yf.start('Josh Hader')              # activate from BN to first open eligible slot
# yf.swap('Josh Hader', 'Clay Holmes')  # swap two players' slots

Before: slot       name positions status
  BN Josh Hader        RP    DTD
  Session: https://baseball.fantasysports.yahoo.com/b1/18397
Moved Josh Hader → P


In [16]:
# Confirm updated roster — check Hader's new slot
updated_roster = yf.roster
hader_after = updated_roster[updated_roster['name'].str.contains('Hader', case=False, na=False)]
print("After:", hader_after[['slot', 'name', 'positions', 'status']].to_string(index=False))
updated_roster

After: slot       name positions status
  BN Josh Hader        RP    DTD


,slot,name,team,positions,status,player_key
0,1B,Rafael Devers,SF,1B,,469.p.10235
1,2B,Xavier Edwards,MIA,"2B,SS",,469.p.11375
2,3B,Matt Chapman,SF,3B,,469.p.10205
3,BN,Victor Scott II,STL,OF,,469.p.61888
4,BN,Josh Hader,HOU,RP,DTD,469.p.10131
5,BN,Michael Wacha,KC,SP,,469.p.9329
6,BN,Andrew Painter,PHI,SP,,469.p.60592
7,BN,Sean Manaea,NYM,"SP,RP",,469.p.9582
8,BN,Will Warren,NYY,SP,,469.p.60295
9,C,Yainer Diaz,HOU,C,,469.p.12435


## Section 10: Trade Proposals

Search for players on other teams before proposing. The `team` argument matches any substring of the opponent's team name.

In [17]:
# Search for a player to check their current team / ownership
yf.search('Julio Rodriguez')

,name,team,positions,status,player_key
0,Julio Rodríguez,SEA,OF,,469.p.11384


In [18]:
# Propose a trade
# yf.trade(
#     give=['Your Player Name'],
#     receive=['Their Player Name'],
#     team='Opponent Team Name',     # case-insensitive substring of their team name
# )

# Multi-player trade
# yf.trade(
#     give=['Player A', 'Player B'],
#     receive=['Player C'],
#     team='Rival Squad',
# )

print('Uncomment above to propose a trade.')

Uncomment above to propose a trade.


## Section 11: Top Available Players Across All Leagues

Fetches the top available batters and pitchers by % rostered for each of your leagues.  
Set `N_PLAYERS` to control how many of each type are returned per league.

In [19]:
N_PLAYERS = 3   # top batters + top pitchers to return per league

top_available = top_available_all_leagues(leagues_df, CREDS_FILE, SEASON, n=N_PLAYERS)
print(f'\nDone \u2014 {len(top_available)} rows ({len(leagues_df)} leagues \u00d7 2 types \u00d7 {N_PLAYERS} players)')
top_available

  Yahoo Prize H2H-Cat 18397...


  Yahoo Prize H2H-Cat 41036...
  Yahoo Prize Roto 15981...


  Yahoo Prize H2H-Cat 68331...
  Yahoo Prize H2H-Cat 148341...


  Yahoo Prize H2H-Cat 3924...
  Yahoo H2H-Cat 690...


  Yahoo Prize Roto 156186...

Done — 43 rows (8 leagues × 2 types × 3 players)


,league,my_team,type,rank,name,team,positions,status,pct_owned
0,Yahoo Prize H2H-Cat 18397,Mother Tuckers 100,batter,1,Kerry Carpenter,DET,OF,,65.0
1,Yahoo Prize H2H-Cat 18397,Mother Tuckers 100,batter,2,Masyn Winn,STL,SS,,57.0
2,Yahoo Prize H2H-Cat 18397,Mother Tuckers 100,batter,3,Alejandro Kirk,TOR,C,,47.0
3,Yahoo Prize H2H-Cat 18397,Mother Tuckers 100,pitcher,1,Roki Sasaki,LAD,SP,,48.0
4,Yahoo Prize H2H-Cat 18397,Mother Tuckers 100,pitcher,2,Yusei Kikuchi,LAA,SP,,25.0
5,Yahoo Prize H2H-Cat 18397,Mother Tuckers 100,pitcher,3,Mitch Keller,PIT,SP,,17.0
6,Yahoo Prize H2H-Cat 41036,Dr. StrangeGlove 100,batter,1,Heliot Ramos,SF,OF,,67.0
7,Yahoo Prize H2H-Cat 41036,Dr. StrangeGlove 100,batter,2,Masyn Winn,STL,SS,,57.0
8,Yahoo Prize H2H-Cat 41036,Dr. StrangeGlove 100,batter,3,Bryson Stott,PHI,"2B,SS",,55.0
9,Yahoo Prize H2H-Cat 41036,Dr. StrangeGlove 100,pitcher,1,Logan Henderson,MIL,SP,DTD,34.0
